# Feedback, coupling, and model-based control

Three bounded comparisons ask what a joint controller must account for: position error, velocity, and the rest of the mechanism. Run one section at a time. These cells expose the existing experiment controllers; they do not implement your walking policy.

**Robot connection:** [position actuators](../../model/spider.xml) turn target angles into forces; [set_targets and step](../../spider/simulation.py) apply those targets; [SupportAwareStanceController](../../spider/controllers.py) uses measured support to adjust them. The torque controllers here remain isolated comparison fixtures. C-1N does not run this computed-torque controller.

**Evidence:** Dated [pendulum](../history/2026-08-08-pendulum-control/README.md), [coupling](../history/2026-08-09-two-link-coupling/README.md), and [model-based control](../history/2026-08-09-model-based-control/README.md) records retain the original findings and adjacent interaction logs. These new notebook cells have not been run to establish new results.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "spider" / "simulation.py").is_file())
sys.path.insert(0, str(ROOT))
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace
from lab import pendulum, two_link_coupling as coupling, model_based_control as dynamics
print("Setup only: no rollout has run.")

## 1. Does velocity feedback change the pendulum response?

**Edit here:** choose one control mode (`constant`, `p`, or `pd`), gains, and a finite duration. Keep the initial angle fixed when comparing modes. The measurement is joint angle and angular velocity against simulation time. The existing controller is [compute_control](../pendulum.py).

In [ ]:
pendulum.CONTROL_MODE = "pd"
pendulum.Kp = 8.0
pendulum.Kd = 1.5
seconds = 4.0
assert 0 < seconds <= 60
model = mujoco.MjModel.from_xml_string(pendulum.XML)
data = mujoco.MjData(model)
data.qpos[0] = 0.2
samples = []
for _ in range(int(np.ceil(seconds / model.opt.timestep))):
    data.ctrl[0] = pendulum.compute_control(data)
    mujoco.mj_step(model, data)
    samples.append((data.time, data.qpos[0], data.qvel[0]))
samples = np.asarray(samples)
fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(samples[:, 0], samples[:, 1], label=pendulum.CONTROL_MODE)
axes[0].axhline(pendulum.TARGET_ANGLE, color="gray", linestyle="--")
axes[0].set_ylabel("angle (rad)")
axes[0].legend()
axes[1].plot(samples[:, 0], samples[:, 2])
axes[1].set(xlabel="simulation time (s)", ylabel="velocity (rad/s)")
fig.tight_layout()

## 2. Does an unpowered elbow move when the shoulder moves?

**Edit here:** joint-2 mode and gravity. Hold pose and shoulder commands fixed. This notebook uses a finite **simulation-time** schedule. The historical viewer uses wall-clock targets, so a new notebook run is a changed protocol, not a reproduction of its timing. Inspect both joint angles and torque commands. [staged_targets and pd](../two_link_coupling.py) retain the original equations.

In [ ]:
joint2_mode = "passive"  # passive, hold, wave
gravity = -9.81
configuration = "elbow-down"
seconds = 12.0
assert joint2_mode in ("passive", "hold", "wave") and 0 < seconds <= 60
model = mujoco.MjModel.from_xml_string(coupling.XML_TEMPLATE.format(
    gravity=gravity, base_height=coupling.BASE_HEIGHT,
    base_pitch_degrees=coupling.BASE_PITCH_DEGREES))
data = mujoco.MjData(model)
q1, q2 = coupling.CONFIGURATIONS[configuration]
data.qpos[:] = (q1, q2)
mujoco.mj_forward(model, data)
samples = []
for _ in range(int(np.ceil(seconds / model.opt.timestep))):
    target1, target2, stage = coupling.staged_targets(data.time, q1, q2, coupling.CONTROL_COUPLING)
    data.ctrl[0] = coupling.pd(target1, data.qpos[0], data.qvel[0], coupling.JOINT1_KP, coupling.JOINT1_KD)
    if joint2_mode == "passive":
        data.ctrl[1] = 0.0
    elif joint2_mode == "hold":
        data.ctrl[1] = coupling.pd(q2, data.qpos[1], data.qvel[1], 2.0, 0.3)
    else:
        data.ctrl[1] = coupling.pd(target2, data.qpos[1], data.qvel[1], coupling.JOINT2_KP, coupling.JOINT2_KD)
    mujoco.mj_step(model, data)
    samples.append((data.time, *data.qpos, *data.ctrl))
samples = np.asarray(samples)
fig, axes = plt.subplots(2, 1, sharex=True)
for j in range(2):
    axes[0].plot(samples[:, 0], samples[:, j + 1], label=f"joint {j + 1}")
    axes[1].plot(samples[:, 0], samples[:, j + 3])
axes[0].set_ylabel("angle (rad)")
axes[0].legend()
axes[1].set(xlabel="simulation time (s)", ylabel="command torque (Nm)")
fig.tight_layout()

## 3. What changes when the controller models gravity and inertia?

**Edit here:** controller and duration. Compare `pd`, `gravity-comp`, `computed-torque`, and `computed-torque-wrong-mass` with the same plant and target. The existing runner measures the last complete target cycle. Its historical convention compares a pre-step target with post-step state; retain that 2 ms alignment limit when interpreting errors. The fixture now returns its existing samples for notebook plots.

In [ ]:
controller = "pd"
seconds = 16.0
assert controller in ("pd", "gravity-comp", "computed-torque", "computed-torque-wrong-mass")
assert 2 * np.pi / dynamics.FREQUENCY <= seconds <= 60
model = dynamics.make_model(controller)
data = mujoco.MjData(model)
data.qpos[:] = dynamics.START
mujoco.mj_forward(model, data)
control_model = dynamics.make_model(controller, link2_mass=(
    dynamics.WRONG_CONTROL_LINK2_MASS if controller == "computed-torque-wrong-mass"
    else dynamics.PLANT_LINK2_MASS))
samples = dynamics.run_headless(SimpleNamespace(controller=controller, duration=seconds),
    model, data, mujoco.MjData(model), control_model, mujoco.MjData(control_model))
fig, axes = plt.subplots(2, 1, sharex=True)
for j in range(2):
    axes[0].plot(samples["time"], samples["error"][:, j], label=f"joint {j + 1}")
    axes[1].plot(samples["time"], samples["tau_total"][:, j])
axes[0].set_ylabel("tracking error (rad)")
axes[0].legend()
axes[1].set(xlabel="simulation time (s)", ylabel="command torque (Nm)")
fig.tight_layout()

## Inspect before carrying the concept into C-1N

Which comparison changed the behavior you predicted? Separate target error, torque demand, and timing error. These fixtures use torque motors; C-1N's policy currently supplies target-angle offsets. Explain the interface difference before transferring a gain or controller. No new interpretation has been supplied here.

For spatial inspection, the original viewers remain in the adjacent fixture modules. Launch them deliberately with `python -m lab.pendulum`, `python -m lab.two_link_coupling`, or `python -m lab.model_based_control --controller overlay`. These are live reruns, not playback of the notebook samples.